# RAG — Retrieval-Augmented Generation
### Chatbot de Joyería con Recuperación Semántica
**Proyecto I: Introducción a LLMs — Facultad de Ciencias, UNAM — Semestre 2026-2**

---

> **Dónde estamos:** El baseline del chatbot funcionó — pero metía el catálogo completo en el prompt. Eso tiene un límite: con cientos de productos, el contexto se desborda. RAG resuelve esto recuperando solo la información relevante para cada pregunta.

### ¿Qué es RAG?

RAG (Lewis et al., 2020) combina dos sistemas:

```
Pregunta del usuario
       │
       ▼
┌─────────────────────┐     ┌────────────────────────────┐
│  RETRIEVER          │────▶│  Base de conocimiento      │
│  (búsqueda semántica│     │  (catálogo, documentos,    │
│   por embeddings)   │◀────│   leyes, PDFs...)          │
└─────────────────────┘     └────────────────────────────┘
       │
       │ Fragmentos relevantes
       ▼
┌─────────────────────┐
│  GENERATOR          │
│  (LLM instruct)     │──▶ Respuesta fundamentada
│  prompt = contexto  │
│         + pregunta  │
└─────────────────────┘
```

**Mapa de la sesión:**
```
Parte 1: Embeddings semánticos — la base del retrieval    (15 min)
Parte 2: Construir la base de conocimiento (vector store) (15 min)
Parte 3: El retriever — búsqueda por similitud            (10 min)
Parte 4: El pipeline RAG completo                         (15 min)
Parte 5: Evaluar y comparar con el baseline               (10 min)
```

**Este mismo patrón aplica al proyecto del asistente contable** — la única diferencia es que la base de conocimiento son documentos fiscales en lugar de un catálogo de productos.

**Infraestructura:** Colab T4 GPU recomendada, pero funciona en CPU (más lento).

In [ ]:
!pip install transformers torch sentence-transformers faiss-cpu --quiet

import torch
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
import faiss
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo: {device}")
print("Librerías listas.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 45.9 MB/s eta 0:00:00


---
## Parte 1: Embeddings Semánticos

La búsqueda por palabras clave exactas falla en lenguaje natural:

```
Usuario: "¿tienen algo plateado para regalo?"
Catálogo: "Aretes de plata 925 con turquesa natural"

Búsqueda por keywords: 0 coincidencias exactas ("plateado" ≠ "plata")
Búsqueda semántica:    alta similitud ("plateado" ≈ "plata")
```

Los **embeddings semánticos** convierten texto en vectores donde la similitud geométrica refleja similitud de significado. Los modelos `sentence-transformers` están optimizados para esto — a diferencia de los embeddings de BERT que vimos antes, estos producen un vector por oración (no por token) y están entrenados con pares similares/disimilares.

In [ ]:
# Modelo de embeddings multilingüe — entiende español e inglés
# paraphrase-multilingual-MiniLM-L12-v2:
#   - 118M parámetros
#   - 50+ idiomas incluyendo español
#   - vectores de 384 dimensiones
#   - muy rápido (diseñado para retrieval)

print("Cargando modelo de embeddings...")
embedding_model = SentenceTransformer(
    'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    device=device
)
print("Listo.")
print(f"Dimensión de los embeddings: {embedding_model.get_sentence_embedding_dimension()}")

Cargando modelo de embeddings...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
# Demostración: similitud semántica en acción

def similitud_coseno(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

pares = [
    # Semánticamente similares
    ("¿tienen algo de plata?",          "Aretes de plata 925",                "similar"),
    ("regalo para mamá",                "pedidos personalizados disponibles", "similar"),
    ("¿cuánto cuesta el envío?",        "costo de envío $120 MXN",           "similar"),
    # Semánticamente distintos
    ("¿tienen algo de plata?",          "política de devoluciones 30 días",  "distinto"),
    ("¿cuánto cuesta el envío?",        "aretes de cobre con baño de oro",   "distinto"),
]

print("Similitud semántica entre preguntas y fragmentos del catálogo:")
print(f"{'Pregunta':>35} | {'Fragmento':>38} | {'Sim':>5} | Tipo")
print("-" * 95)

for pregunta, fragmento, tipo in pares:
    emb_a = embedding_model.encode(pregunta)
    emb_b = embedding_model.encode(fragmento)
    sim = similitud_coseno(emb_a, emb_b)
    marca = "✓" if tipo == "similar" else "·"
    print(f"{pregunta:>35} | {fragmento:>38} | {sim:.3f} | {marca} {tipo}")

print()
print("Observar: 'plateado' y 'plata' tienen alta similitud aunque no comparten")
print("ningún token. El modelo aprendió la relación semántica durante pretraining.")

---
## Parte 2: Construir la Base de Conocimiento

La base de conocimiento es el catálogo dividido en **fragmentos** (chunks). Cada fragmento es una unidad recuperable — en el proyecto real, estos fragmentos vendrían de los documentos del negocio.

**¿Por qué fragmentar?**
- Un fragmento pequeño y específico es más fácil de recuperar correctamente
- El modelo puede concentrarse en la información relevante sin ruido
- La ventana de contexto del LLM no se desborda

**FAISS** (Facebook AI Similarity Search) es la librería estándar para búsqueda por similitud en vectores. Permite encontrar los $k$ vectores más cercanos a una consulta en milisegundos, incluso con millones de vectores.

In [ ]:
# Catálogo dividido en fragmentos recuperables
# Cada fragmento es autosuficiente — contiene toda la info necesaria

CHUNKS = [
    # ── ARETES ────────────────────────────────────────────────────────────
    "Aretes de plata 925 con turquesa natural. Precio: $650 MXN. "
    "Disponibles en dos colores: azul y verde. Material: plata de ley 925 con piedra turquesa natural.",

    "Aretes de cobre con baño de oro y piedra ojo de tigre. Precio: $480 MXN. "
    "Stock muy limitado, solo quedan 3 pares disponibles. Material: cobre con baño de oro y piedra semipreciosa.",

    "Aretes minimalistas de plata lisa con forma de luna. Precio: $320 MXN. "
    "Siempre disponibles, diseño sencillo y elegante. Son los aretes más económicos del catálogo.",

    "Aretes largos de plata con cuentas de obsidiana. Precio: $580 MXN. "
    "Diseño artesanal con piedra obsidiana mexicana. Disponibles en stock normal.",

    # ── COLLARES ──────────────────────────────────────────────────────────
    "Collar de plata con dije de obsidiana tallada a mano. Precio: $1,200 MXN. "
    "Cada pieza es única, tallada individualmente por artesanos. Material: plata 925 y obsidiana.",

    "Collar de cuarzo rosa en cadena de plata 925. Precio: $980 MXN. "
    "Disponible en stock normal. Ideal para regalo romántico. Piedra: cuarzo rosa natural.",

    "Gargantilla de cobre trenzado. Precio: $550 MXN. "
    "Disponible en tres grosores: delgado, mediano y grueso. Material: cobre trabajado a mano.",

    "Collar choker de plata con piedra labradorita. Precio: $1,100 MXN. "
    "Stock muy limitado, solo 2 piezas disponibles. La labradorita cambia de color con la luz.",

    # ── PULSERAS ──────────────────────────────────────────────────────────
    "Pulsera de plata con charms artesanales: colibrí, luna y sol. Precio: $750 MXN. "
    "Completamente personalizable — puedes elegir los charms. Disponible bajo pedido.",

    "Brazalete de cobre martillado. Precio: $420 MXN. "
    "Talla única ajustable, se adapta a cualquier muñeca. Material: cobre trabajado con técnica de martillado.",

    "Pulsera de plata con piedra de luna (moonstone). Precio: $890 MXN. "
    "Disponible en stock normal. La piedra de luna tiene un brillo azulado característico.",

    # ── ENVÍOS ────────────────────────────────────────────────────────────
    "Información de envíos: hacemos envíos a toda la República Mexicana. "
    "Costo de envío: $120 MXN. Envío GRATIS en compras mayores a $1,500 MXN. "
    "Tiempo de entrega: 3-5 días hábiles en CDMX, 5-7 días en el resto del país.",

    # ── PAGOS ─────────────────────────────────────────────────────────────
    "Métodos de pago aceptados: tarjeta de crédito o débito, transferencia bancaria (SPEI) y PayPal. "
    "No manejamos efectivo ni depósitos en tiendas de conveniencia.",

    # ── GARANTÍA Y DEVOLUCIONES ───────────────────────────────────────────
    "Garantía: 6 meses en piezas de plata 925 contra defectos de fabricación. "
    "Política de devolución: 30 días en producto sin uso y en empaque original. "
    "Si el producto llega dañado, lo reponemos sin costo adicional.",

    # ── PEDIDOS PERSONALIZADOS ────────────────────────────────────────────
    "Pedidos personalizados: sí trabajamos piezas a la medida. "
    "Tiempo adicional de 7-10 días hábiles. Se requiere 50% de anticipo. "
    "Puedes pedir grabado de nombre, fechas, o diseños especiales.",

    # ── MATERIALES ────────────────────────────────────────────────────────
    "Materiales que trabajamos: plata 925 (plata de ley), cobre artesanal, "
    "y piedras semipreciosas como turquesa, obsidiana, cuarzo rosa, labradorita, "
    "ojo de tigre y moonstone. No trabajamos con oro macizo ni plata baja ley.",

    # ── CONTACTO ──────────────────────────────────────────────────────────
    "Contacto y redes sociales del Taller Luna Plata: "
    "Instagram: @tallerlunaplata | Email: contacto@tallerlunaplata.com | "
    "WhatsApp: 55-1234-5678. Horario de atención: lunes a viernes 10:00-18:00.",
]

print(f"Base de conocimiento: {len(CHUNKS)} fragmentos")
print(f"Longitud promedio: {np.mean([len(c) for c in CHUNKS]):.0f} caracteres")
print(f"Longitud máxima:   {max(len(c) for c in CHUNKS)} caracteres")
print()
print("Fragmento más largo:")
print(" ", max(CHUNKS, key=len))

In [ ]:
# Paso 1: Convertir todos los chunks a vectores
print("Codificando fragmentos...")
chunk_embeddings = embedding_model.encode(
    CHUNKS,
    show_progress_bar=True,
    batch_size=16,
    normalize_embeddings=True,   # normalizar para que dot product = similitud coseno
)
print(f"\nMatrix de embeddings: {chunk_embeddings.shape}")
print(f"  {len(CHUNKS)} fragmentos × {chunk_embeddings.shape[1]} dimensiones")

# Paso 2: Construir el índice FAISS
dim = chunk_embeddings.shape[1]

# IndexFlatIP: búsqueda exacta por producto interno (= coseno si están normalizados)
# Para millones de vectores se usaría IndexIVFFlat (aproximado pero más rápido)
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings.astype(np.float32))

print(f"\nÍndice FAISS construido.")
print(f"Vectores indexados: {index.ntotal}")
print(f"Tipo de índice: IndexFlatIP (búsqueda exacta por similitud coseno)")

In [ ]:
# Visualizar el espacio de embeddings con PCA (reducción a 2D)
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
coords_2d = pca.fit_transform(chunk_embeddings)

# Categorías para colorear
categorias = [
    'aretes', 'aretes', 'aretes', 'aretes',
    'collares', 'collares', 'collares', 'collares',
    'pulseras', 'pulseras', 'pulseras',
    'logística', 'logística', 'logística', 'logística', 'logística', 'logística',
]

colores_cat = {'aretes': '#1565C0', 'collares': '#C62828',
               'pulseras': '#2E7D32', 'logística': '#E65100'}

fig, ax = plt.subplots(figsize=(10, 7))
for cat in set(categorias):
    idx = [i for i, c in enumerate(categorias) if c == cat]
    ax.scatter(coords_2d[idx, 0], coords_2d[idx, 1],
               c=colores_cat[cat], s=120, label=cat, zorder=3)

# Etiquetar algunos puntos
etiquetas_cortas = [
    'Aretes turquesa', 'Aretes cobre-oro', 'Aretes luna', 'Aretes obsidiana',
    'Collar obsidiana', 'Collar cuarzo', 'Gargantilla', 'Collar labradorita',
    'Pulsera charms', 'Brazalete cobre', 'Pulsera moonstone',
    'Envíos', 'Pagos', 'Garantía', 'Personalizado', 'Materiales', 'Contacto',
]
for i, etiq in enumerate(etiquetas_cortas):
    ax.annotate(etiq, coords_2d[i], textcoords='offset points',
                xytext=(5, 3), fontsize=7, alpha=0.8)

ax.set_title('Espacio de embeddings del catálogo (PCA 2D)\n'
             'Fragmentos similares semánticamente están cerca', fontsize=12, fontweight='bold')
ax.legend(title='Categoría', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Los fragmentos de la misma categoría (aretes, collares, etc.) tienden")
print("a agruparse en el espacio vectorial — el modelo captura la estructura semántica.")

---
## Parte 3: El Retriever — Búsqueda por Similitud

In [ ]:
def recuperar(pregunta, k=3, verbose=True):
    """
    Dado una pregunta, recupera los k fragmentos más relevantes.

    Pipeline:
    1. Codificar la pregunta → vector de 384 dims
    2. Buscar los k vecinos más cercanos en el índice FAISS
    3. Devolver los fragmentos correspondientes con sus scores
    """
    # Codificar la pregunta
    q_embedding = embedding_model.encode(
        [pregunta],
        normalize_embeddings=True
    ).astype(np.float32)

    # Buscar en FAISS
    scores, indices = index.search(q_embedding, k)

    resultados = []
    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:
            resultados.append({'chunk': CHUNKS[idx], 'score': float(score), 'idx': int(idx)})

    if verbose:
        print(f"Pregunta: '{pregunta}'")
        print(f"Top {k} fragmentos recuperados:")
        for i, r in enumerate(resultados, 1):
            print(f"  {i}. [score={r['score']:.4f}] {r['chunk'][:80]}...")
        print()

    return resultados


# Probar el retriever con distintos tipos de preguntas
preguntas_test = [
    "¿tienen algo plateado para regalo?",
    "¿cuánto cuesta mandar un pedido a Monterrey?",
    "quiero una pulsera personalizada con el nombre de mi novia",
    "¿qué pasa si el producto llega dañado?",
    "¿tienen anillos de oro?",  # fuera del catálogo
]

print("=" * 65)
print("Prueba del Retriever")
print("=" * 65)
for pregunta in preguntas_test:
    recuperar(pregunta, k=2)

In [ ]:
# Umbral de relevancia: no recuperar si la similitud es muy baja
# Esto evita que el modelo reciba fragmentos irrelevantes

UMBRAL_SIMILITUD = 0.30  # ajustar según el dominio

def recuperar_con_umbral(pregunta, k=3):
    resultados = recuperar(pregunta, k=k, verbose=False)
    filtrados = [r for r in resultados if r['score'] >= UMBRAL_SIMILITUD]
    return filtrados


# Ver cómo varía el score según relevancia
print("Scores de recuperación para preguntas dentro y fuera del catálogo:")
print(f"{'Pregunta':>50} | {'Score top-1':>12} | {'En catálogo'}")
print("-" * 80)

ejemplos = [
    ("¿cuánto cuestan los aretes de plata?",         True),
    ("¿tienen envío gratis?",                         True),
    ("¿hacen devoluciones?",                          True),
    ("¿tienen anillos de compromiso en diamante?",   False),
    ("¿cuál es el precio del dólar hoy?",            False),
    ("¿tienen descuentos por volumen?",               False),
]

for pregunta, en_catalogo in ejemplos:
    resultados = recuperar(pregunta, k=1, verbose=False)
    score = resultados[0]['score'] if resultados else 0
    sobre_umbral = "✓" if score >= UMBRAL_SIMILITUD else "✗ (bajo umbral)"
    print(f"{pregunta:>50} | {score:>12.4f} | {sobre_umbral}")

print()
print(f"Umbral actual: {UMBRAL_SIMILITUD}")
print("Si el score es bajo → el chatbot debe decir que no tiene esa información.")
print("Ajusta el umbral según los resultados de tu dominio específico.")

---
## Parte 4: El Pipeline RAG Completo

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Cargando {MODEL_ID}...")

gen_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
gen_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
    device_map='auto' if device == 'cuda' else None,
)
if device == 'cpu':
    gen_model = gen_model.to('cpu')
gen_model.eval()
print("Listo.")

In [ ]:
SYSTEM_BASE = """Eres Luna, la asistente virtual del Taller Luna Plata, una tienda de joyería artesanal mexicana.

Tu personalidad:
- Amable, cálida y apasionada por la joyería artesanal
- Siempre respondes en español
- Honesta: si algo no está en la información disponible, lo dices claramente

Tus reglas:
- Usa ÚNICAMENTE la información del contexto para responder
- No inventes productos, precios ni políticas que no aparezcan en el contexto
- Si la pregunta no puede responderse con el contexto, di: "No tengo esa información,
  pero puedes contactarnos en contacto@tallerlunaplata.com"
- Respuestas concisas: máximo 3-4 oraciones"""


def generar_respuesta_rag(pregunta, contexto_chunks, historial=None, max_tokens=200):
    """
    Genera una respuesta usando RAG:
    1. El contexto recuperado se inyecta en el system prompt
    2. El modelo genera basándose SOLO en ese contexto
    """
    if historial is None:
        historial = []

    # Construir el contexto como texto
    if contexto_chunks:
        contexto_texto = "\n\n".join(
            f"[{i+1}] {chunk['chunk']}" for i, chunk in enumerate(contexto_chunks)
        )
        system = SYSTEM_BASE + f"\n\nINFORMACIÓN DISPONIBLE:\n{contexto_texto}"
    else:
        # Sin contexto relevante — el modelo debe reconocerlo
        system = SYSTEM_BASE + "\n\nINFORMACIÓN DISPONIBLE: No se encontró información relevante para esta consulta."

    mensajes = [{"role": "system", "content": system}]
    mensajes += historial
    mensajes.append({"role": "user", "content": pregunta})

    texto = gen_tokenizer.apply_chat_template(
        mensajes, tokenize=False, add_generation_prompt=True
    )
    inputs = gen_tokenizer(texto, return_tensors='pt').to(gen_model.device)
    n_input = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = gen_model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=gen_tokenizer.eos_token_id,
        )

    respuesta = gen_tokenizer.decode(
        outputs[0][n_input:], skip_special_tokens=True
    ).strip()
    return respuesta


def chatbot_rag(pregunta, historial=None, k=3, verbose=False):
    """
    Pipeline RAG completo:
    1. Retrieve: buscar fragmentos relevantes
    2. Augment:  construir prompt con contexto
    3. Generate: producir respuesta
    """
    if historial is None:
        historial = []

    # RETRIEVE
    chunks_relevantes = recuperar_con_umbral(pregunta, k=k)

    if verbose:
        print(f"[Retrieval] {len(chunks_relevantes)}/{k} fragmentos sobre el umbral")
        for r in chunks_relevantes:
            print(f"  score={r['score']:.3f}: {r['chunk'][:60]}...")

    # AUGMENT + GENERATE
    respuesta = generar_respuesta_rag(pregunta, chunks_relevantes, historial)
    return respuesta, chunks_relevantes


# Prueba rápida
print("Prueba del pipeline RAG:")
print()
pregunta_test = "¿Cuánto cuestan los aretes de plata con turquesa?"
respuesta, chunks = chatbot_rag(pregunta_test, verbose=True)
print(f"👤 {pregunta_test}")
print(f"🤖 {respuesta}")

In [ ]:
# Batería completa de pruebas

preguntas_bateria = [
    ("¿qué aretes tienen de plata?",                       "producto"),
    ("¿cuánto cuesta enviar a Guadalajara?",               "logística"),
    ("quiero regalarle algo a mi mamá, ¿qué me recomiendas?", "recomendación"),
    ("¿puedo pagar con transferencia?",                    "pago"),
    ("¿hacen pulseras con nombres grabados?",              "personalización"),
    ("¿qué garantía tienen los productos de plata?",       "garantía"),
    ("¿tienen algo menos de 500 pesos?",                   "precio"),
    ("¿tienen anillos de brillantes?",                     "fuera de catálogo"),
    ("¿cuál es su horario de atención?",                   "contacto"),
    ("¿el collar de obsidiana es único o tienen varios?",  "disponibilidad"),
]

print("=" * 65)
print("BATERÍA DE PRUEBAS — Chatbot RAG")
print("=" * 65)

resultados_rag = []
for pregunta, categoria in preguntas_bateria:
    respuesta, chunks = chatbot_rag(pregunta, k=3)
    resultados_rag.append({
        'pregunta': pregunta,
        'categoria': categoria,
        'respuesta': respuesta,
        'n_chunks': len(chunks),
        'score_top': chunks[0]['score'] if chunks else 0,
    })
    print(f"\n[{categoria.upper()}]")
    print(f"👤 {pregunta}")
    print(f"🤖 {respuesta}")

In [ ]:
# Conversación multi-turno con RAG
# El historial persiste entre turnos; el retrieval se hace en cada turno

print("=" * 65)
print("CONVERSACIÓN MULTI-TURNO")
print("=" * 65)
print()

historial = []
conversacion = [
    "Hola, estoy buscando un regalo para mi novia.",
    "Le gustan las cosas de plata con piedras. ¿Qué me recomiendas?",
    "¿Cuánto cuesta ese collar?",
    "¿Me lo pueden enviar a Monterrey y cuánto tarda?",
    "¿Puedo pagar con PayPal?",
]

for turno, mensaje in enumerate(conversacion, 1):
    print(f"Turno {turno}:")
    print(f"👤 {mensaje}")

    respuesta, chunks = chatbot_rag(mensaje, historial=historial, k=2)
    print(f"🤖 {respuesta}")
    score_top = f"{chunks[0]['score']:.3f}" if chunks else "0"
    print(f"   [chunks recuperados: {len(chunks)}, score top: {score_top}]")
    print()

    # Actualizar historial
    historial.append({"role": "user",      "content": mensaje})
    historial.append({"role": "assistant", "content": respuesta})

    # Limitar historial para no desbordar el contexto
    if len(historial) > 8:
        historial = historial[-8:]


---
## Parte 5: Evaluar y Comparar con el Baseline

In [ ]:
# Comparación cualitativa: baseline (catálogo completo en prompt) vs RAG
# La diferencia clave es qué información recibe el modelo

print("Comparación: Baseline vs RAG")
print("=" * 65)

# Calcular tokens de contexto en cada enfoque
CATALOGO_COMPLETO = "\n".join(CHUNKS)  # equivalente al catálogo del baseline
tokens_baseline = len(gen_tokenizer.encode(CATALOGO_COMPLETO))

# Para RAG: tokens promedio de los chunks recuperados
tokens_rag_promedio = np.mean([
    len(gen_tokenizer.encode(" ".join(c['chunk'] for c in r['n_chunks'] * [{'chunk': CHUNKS[0]}])))
    if r['n_chunks'] > 0 else 0
    for r in resultados_rag
])
# Simplificado: tokens de 3 chunks promedio
muestra_chunks = " ".join(CHUNKS[:3])
tokens_rag_aprox = len(gen_tokenizer.encode(muestra_chunks))

print(f"\nTokens de contexto por consulta:")
print(f"  Baseline (catálogo completo):  {tokens_baseline:>6,} tokens")
print(f"  RAG (3 chunks recuperados):    {tokens_rag_aprox:>6,} tokens")
print(f"  Reducción:                     {(1 - tokens_rag_aprox/tokens_baseline)*100:.0f}%")
print()
print("Implicaciones:")
print("  Menos tokens = respuesta más rápida y menor costo de cómputo")
print("  Con catálogos de 500+ productos el baseline falla; RAG escala")
print("  El modelo puede concentrarse en la información realmente relevante")

# Gráfica comparativa
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Baseline vs RAG', fontsize=13, fontweight='bold')

# Tokens de contexto
axes[0].bar(['Baseline\n(catálogo completo)', 'RAG\n(3 chunks)'],
            [tokens_baseline, tokens_rag_aprox],
            color=['#90CAF9', '#1565C0'], edgecolor='#333', linewidth=1.5)
axes[0].set_ylabel('Tokens de contexto por consulta')
axes[0].set_title('Eficiencia de contexto')
for i, v in enumerate([tokens_baseline, tokens_rag_aprox]):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold', fontsize=11)
axes[0].grid(True, alpha=0.3, axis='y')

# Escalabilidad
n_productos = [10, 50, 100, 500, 1000, 5000]
tokens_baseline_escala = [n * 60 for n in n_productos]  # ~60 tokens por chunk
tokens_rag_escala = [3 * 60 for _ in n_productos]        # siempre 3 chunks

axes[1].plot(n_productos, tokens_baseline_escala, 'o-',
             color='#90CAF9', linewidth=2, label='Baseline (catálogo completo)')
axes[1].plot(n_productos, tokens_rag_escala, 'o-',
             color='#1565C0', linewidth=2, label='RAG (k=3 chunks)')
axes[1].axhline(4096, color='red', linestyle='--', alpha=0.5, label='Límite ctx típico (4K)')
axes[1].set_xlabel('Número de productos en el catálogo')
axes[1].set_ylabel('Tokens de contexto')
axes[1].set_title('Escalabilidad')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import json
from datetime import datetime

tarjeta = {
    "fecha": datetime.now().strftime("%Y-%m-%d"),
    "experimento": "rag-v1",
    "proyecto": "Chatbot joyería artesanal — Taller Luna Plata",
    "arquitectura": "RAG (Retrieval-Augmented Generation)",
    "componentes": {
        "retriever": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        "vector_store": "FAISS IndexFlatIP",
        "generator": "Qwen/Qwen2.5-1.5B-Instruct",
    },
    "parametros": {
        "k_chunks": 3,
        "umbral_similitud": UMBRAL_SIMILITUD,
        "embedding_dim": 384,
        "n_chunks_base_conocimiento": len(CHUNKS),
    },
    "metricas": {
        "tokens_contexto_baseline": tokens_baseline,
        "tokens_contexto_rag_aprox": tokens_rag_aprox,
        "reduccion_contexto_pct": round((1 - tokens_rag_aprox/tokens_baseline)*100, 1),
        "score_rubrica_manual": "[COMPLETAR]",
        "tasa_alucinacion": "[COMPLETAR]",
    },
    "siguiente_experimento": "Ampliar base de conocimiento con FAQ reales + evaluar con rúbrica"
}

print("Tarjeta de experimento:")
print(json.dumps(tarjeta, ensure_ascii=False, indent=2))
print()
print("Guardar como: experimentos/rag_chatbot_v1.json")

---
## Resumen y Siguiente Paso

**Lo que construimos:**

| Componente | Tecnología | Por qué |
|:---|:---|:---|
| Embeddings | `paraphrase-multilingual-MiniLM-L12-v2` | Multilingüe, rápido, 384 dims |
| Vector store | FAISS `IndexFlatIP` | Búsqueda exacta, simple, sin servidor |
| Generator | `Qwen2.5-1.5B-Instruct` | Sigue instrucciones en español |
| Umbral | `score ≥ 0.30` | Detecta preguntas fuera del catálogo |

**El mismo patrón para el proyecto del asistente contable:**
```python
# Solo cambia la fuente de los chunks:
# En lugar del catálogo de joyería, cargar los artículos del SAT

chunks_ley = extraer_chunks_de_pdf("ley_isr_2025.pdf", chunk_size=300)
chunks_sat = extraer_chunks_de_pdf("codigo_fiscal.pdf", chunk_size=300)
CHUNKS = chunks_ley + chunks_sat
# El resto del pipeline es idéntico
```

**Próximo paso para este proyecto:**
1. Evaluar 20 preguntas con la rúbrica de la sesión anterior
2. Identificar preguntas donde el retrieval falla (score bajo pero respuesta esperada existe)
3. Ajustar el tamaño y estructura de los chunks según los errores encontrados